In [1]:
! pip install ultralytics

In [2]:
import os
from torchvision.datasets import VOCDetection
import torch

from ultralytics import YOLO

In [3]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

In [4]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

In [5]:
dataset = VOCDetection("./", year='2012', image_set='train', download=True)

Convert VOC to YOLO format

In [6]:
# YOLO format folder structure
os.makedirs("VOC2012_YOLO/images/train", exist_ok=True)
os.makedirs("VOC2012_YOLO/labels/train", exist_ok=True)

for idx in range(len(dataset)):
    img, target = dataset[idx]
    img_id = target['annotation']['filename'].split('.')[0]
    
    # Save image
    img.save(f"VOC2012_YOLO/images/train/{img_id}.jpg")
    
    # Write YOLO annotation
    h = int(target['annotation']['size']['height'])
    w = int(target['annotation']['size']['width'])
    
    with open(f"VOC2012_YOLO/labels/train/{img_id}.txt", 'w') as f:
        for obj in target['annotation']['object']:
            if isinstance(obj, dict):  # sometimes object is a dict
                cls_name = obj['name']
                cls_id = VOC_CLASSES.index(cls_name)
                bbox = obj['bndbox']
                # Normalize x_center, y_center, width, height
                x_center = (int(bbox['xmin']) + int(bbox['xmax'])) / 2 / w
                y_center = (int(bbox['ymin']) + int(bbox['ymax'])) / 2 / h
                bw = (int(bbox['xmax']) - int(bbox['xmin'])) / w
                bh = (int(bbox['ymax']) - int(bbox['ymin'])) / h
                f.write(f"{cls_id} {x_center} {y_center} {bw} {bh}\n")

In [7]:
def create_yaml():
    yaml_path = f"VOC2012_YOLO/voc2012.yaml"
    with open(yaml_path, "w") as f:
        f.write(
f"""
path: VOC2012_YOLO
train: images/train
val: images/train
nc: 20
names: {VOC_CLASSES}
"""
        )
    return yaml_path

In [8]:
yaml_path = create_yaml()

In [9]:
# Load a pre-trained YOLOv8 model
model = YOLO("yolov8n.pt")  # small model for quick training

# Train on VOC 2012
model.train(
    data="VOC2012_YOLO/voc2012.yaml",
    epochs=3,
    imgsz=640,
    batch=16,
    device=device,
    amp=False,
    workers=2,
    val=False
)

engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VOC2012_YOLO/voc2012.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x139776850>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043

In [10]:
model.predict("VOC2012_YOLO/images/train/2008_000008.jpg", conf=0.25, show_boxes=True, show_labels=True, show_conf=True, save=True, save_dir="yolo/runs/detect")


image 1/1 /Users/wick/AI_fundamentals/examples/3_computer_vision/VOC2012_YOLO/images/train/2008_000008.jpg: 576x640 2 boats, 2 horses, 2 persons, 39.8ms
Speed: 24.8ms preprocess, 39.8ms inference, 6.9ms postprocess per image at shape (1, 3, 576, 640)
Results saved to /Users/wick/AI_fundamentals/examples/3_computer_vision/yolo/runs/detect


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'aeroplane', 1: 'bicycle', 2: 'bird', 3: 'boat', 4: 'bottle', 5: 'bus', 6: 'car', 7: 'cat', 8: 'chair', 9: 'cow', 10: 'diningtable', 11: 'dog', 12: 'horse', 13: 'motorbike', 14: 'person', 15: 'pottedplant', 16: 'sheep', 17: 'sofa', 18: 'train', 19: 'tvmonitor'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [249, 205, 204],
         [249, 205, 204],
         [249, 205, 204]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [250, 206, 205],
         [250, 206, 205],
         [250, 206, 205]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [250, 209, 207],
         [250, 209, 207],
         [250, 209, 207]],
 
        ...,
 
        [[184, 198, 210],
     